# Phase 5: fine-tune Reson8 Rumble-2-Flash with LoRA, then evaluate it like the zero-shot run

Same recipe as whisper-small/medium (notebooks 03, 05): LoRA r32/alpha64 on attention and feed-forward layers of
encoder and decoder, LR 1e-4, effective batch 16, dev WER every 100 steps on the segmented pieces, early stopping,
best checkpoint by dev WER, SpecAugment. Training path: `cyasr/train_qwen.py` (`backend: qwen3_asr`). The prompt is
the matched inference prompt (Greek forced, numbers as words, fillers kept), so training and evaluation agree.

Then the fair evaluation, identical to the zero-shot run in notebook 07: 280 full-length dev clips, one pass per
clip, greedy, batch 1 (`configs/eval_rumble-2-flash_lora.yaml`).

Run on an **L4**. Needs `MyDrive/cyasr/cyasr_bundle.zip` (data), `cyasr_code.zip` (current code) and `segments.zip`.
Training and evaluation run in the background and write to Drive, so a dropped browser connection does not stop
them; the last cell waits for them, prints the results and (if `AUTO_DISCONNECT`) releases the GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
CY = '/content/drive/MyDrive/cyasr'
DRIVE_RUNS = f'{CY}/runs'
TRAIN_CONFIG = 'configs/finetune_rumble-2-flash.yaml'
EVAL_CONFIG = 'configs/eval_rumble-2-flash_lora.yaml'

!rm -rf /content/cyasr && mkdir -p /content/cyasr
!unzip -q {CY}/cyasr_bundle.zip -d /content/cyasr
!unzip -q -o {CY}/cyasr_code.zip -d /content/cyasr   # newer code on top of the bundle
%cd /content/cyasr
!ls cyasr/train_qwen.py {TRAIN_CONFIG} {EVAL_CONFIG}  # fails if cyasr_code.zip is old
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao   # Colab's old torchao breaks peft; not used
# The segmented pieces as one zip (one large file copies from Drive in about a minute)
!cp {CY}/segments.zip /content/segments.zip && mkdir -p data && unzip -q -o /content/segments.zip -d data/
!ls data/segments/train | wc -l  # expect 820
!ls data/segments/dev | wc -l    # expect 407

In [ ]:
# Smoke test: 4 steps, 8 dev pieces, saved locally. Check the example input/target lines (prompt with
# <nodigits><filler><verbatim> and 'language Greek<asr_text>', target = the Cypriot text + <|im_end|>)
# and peak_gpu_memory_gb in the timing line (the L4 has 23 GB).
import yaml, json, time, subprocess, os
!rm -rf runs/finetune_smoke
cfg = yaml.safe_load(open(TRAIN_CONFIG))
cfg.update(run_name='smoke', max_steps=4, eval_steps=2, warmup_steps=1, max_eval_samples=8)
yaml.safe_dump(cfg, open('/content/smoke.yaml', 'w'), allow_unicode=True)
!python -m cyasr.train --config /content/smoke.yaml 2>&1 | grep -v -i warn | grep -E "example|trainable|pieces|eval_wer|timing|Error|error" 

In [ ]:
# The real run in the background: training (resume on, checkpoints on Drive), then the fair dev evaluation.
# After a disconnect: rerun the cells above (skip the smoke test) and this one; training continues from the
# last checkpoint. The log is on Drive.
LOG = f'{DRIVE_RUNS}/rumble_lora.log'
cfg = yaml.safe_load(open(TRAIN_CONFIG)); cfg.update(output_dir=DRIVE_RUNS, resume=True)
yaml.safe_dump(cfg, open('/content/train.yaml', 'w'), allow_unicode=True)
ev = yaml.safe_load(open(EVAL_CONFIG)); ev.update(output_dir=DRIVE_RUNS)
yaml.safe_dump(ev, open('/content/eval.yaml', 'w'), allow_unicode=True)
F = 'grep --line-buffered -v -i -E "warn|Loading weights|it/s]"'
open('/content/run_rumble.sh', 'w').write(f'''
echo ">>> train start $(date +%T)"
python -m cyasr.train --config /content/train.yaml 2>&1 | {F}
echo ">>> train end $(date +%T)"
echo ">>> eval start $(date +%T)"
python -m cyasr.run_baseline --config /content/eval.yaml 2>&1 | {F}
echo ">>> eval end $(date +%T)"
echo ">>> ALL DONE"
''')
!mkdir -p {DRIVE_RUNS}
proc = subprocess.Popen(f'cd /content/cyasr && PYTHONUNBUFFERED=1 nohup bash /content/run_rumble.sh > {LOG} 2>&1', shell=True)
print('started; log:', LOG)

In [ ]:
# Progress (rerun any time)
!grep -E ">>>|eval_wer_basic|\\[dev\\] [0-9]+/280|WER raw|Error" {LOG} | tail -8; date +%T

In [ ]:
# Waits for training + evaluation, prints the results, then releases the GPU if AUTO_DISCONNECT.
AUTO_DISCONNECT = True
running = lambda: subprocess.run('pgrep -f "[r]un_rumble.sh|[c]yasr.train|[c]yasr.run_baseline"', shell=True).returncode == 0
while running():
    time.sleep(120)
try:
    RUN = f'{DRIVE_RUNS}/finetune_rumble-2-flash_lora'
    log = json.load(open(f'{RUN}/train_log.json'))
    train_loss = {e['step']: e['loss'] for e in log if 'loss' in e}
    print('step  train_loss  dev_loss  WER_basic  WER_nodiac  CER_nodiac   (dev = segmented pieces)')
    for e in log:
        if 'eval_wer_basic' in e:
            print(f"{e['step']:4}  {train_loss.get(e['step'], float('nan')):10.3f}  {e['eval_loss']:8.3f}  "
                  f"{e['eval_wer_basic']:9.1%}  {e['eval_wer_nodiac']:10.1%}  {e['eval_cer_nodiac']:10.1%}")
    print(json.dumps(json.load(open(f'{RUN}/timing.json'))))

    import jiwer
    from cyasr.normalize import normalize
    bucket = lambda d: '<=30s' if d <= 30 else ('30-33s' if d <= 33 else '>33s')
    rows = [('zero-shot whisper-medium', 'baseline_whisper-medium_fallback_dev'),
            ('fine-tuned whisper-medium +rule', 'whisper-medium_lora_fallback_ns0.6_dev'),
            ('zero-shot rumble (matched)', 'baseline_rumble-2-flash_matched_greedy_dev'),
            ('fine-tuned rumble', 'rumble-2-flash_lora_greedy_dev')]
    print(f"\n{'model (full-length dev clips)':32} {'WER basic':>9} {'nodiac':>7} {'CER nod':>7} {'<=30s':>6} {'30-33s':>7} {'>33s':>6} {'halluc':>6}")
    for label, name in rows:
        path = f'{DRIVE_RUNS}/{name}'
        if not os.path.exists(f'{path}/metrics.json'):
            print(f'{label:32} missing'); continue
        m = json.load(open(f'{path}/metrics.json')); lv = m['levels']
        errs, words = {}, {}
        for r in map(json.loads, open(f'{path}/predictions.jsonl', encoding='utf-8')):
            ref, hyp = normalize(r['ref'], 'nodiac'), normalize(r['hyp'], 'nodiac')
            o = jiwer.process_words(ref, hyp); b = bucket(r['duration_s'])
            errs[b] = errs.get(b, 0) + o.substitutions + o.deletions + o.insertions
            words[b] = words.get(b, 0) + len(ref.split())
        w = {b: errs[b] / words[b] for b in errs}
        print(f"{label:32} {lv['basic']['wer']:9.1%} {lv['nodiac']['wer']:7.1%} {lv['nodiac']['cer']:7.1%} "
              f"{w['<=30s']:6.1%} {w['30-33s']:7.1%} {w['>33s']:6.1%} {m['hallucinations']:6}")
except Exception as e:
    print('results failed:', repr(e))
print(open(LOG).read()[-1200:])
if AUTO_DISCONNECT:
    from google.colab import runtime
    drive.flush_and_unmount()
    runtime.unassign()